In [2]:
import pyspark

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.master( "local[7]" ) \
    .appName('Spark') \
    .getOrCreate()

In [22]:
spark = SparkSession.builder \
    .appName("myApp") \
    .config("spark.sql.shuffle.partitions", 10) \
    .getOrCreate()

In [10]:
print("App Name:", spark.sparkContext.appName)
#print("Master:" spark.sparkContext.master)

SyntaxError: invalid syntax (<ipython-input-10-cbffb50993d3>, line 2)

In [3]:
sc = spark.sparkContext

In [27]:
rdd = spark.sparkContext.parallelize([1,2,3,4])
print(rdd.map(lambda x:x * 2).collect())

[2, 4, 6, 8]


In [7]:
df = spark.createDataFrame([(1,"A"),(2,"B"),(3,"C")], ["id", "name"])

# Transformations
filtered_df = df.filter(df.id > 1)
new_col_df = df.withColumn("id_squared", df.id * df.id)

# Execution on Actions
filtered_df.show()
new_col_df.show()

+---+----+
| id|name|
+---+----+
|  2|   B|
|  3|   C|
+---+----+

+---+----+----------+
| id|name|id_squared|
+---+----+----------+
|  1|   A|         1|
|  2|   B|         4|
|  3|   C|         9|
+---+----+----------+



In [11]:
df = spark.range(1, 6)

# Actions
row_count = df.count()
all_rows = df.collect()

print("Row Count:", row_count)
print("Rows:", all_rows)

Row Count: 5
Rows: [Row(id=1), Row(id=2), Row(id=3), Row(id=4), Row(id=5)]


In [12]:
df = spark.range(1,10)

# Lazy Transformations
df2 = df.filter("id > 5").withColumn("double_id", df.id * 2)

# Execution happens here
df2.show()

+---+---------+
| id|double_id|
+---+---------+
|  6|       12|
|  7|       14|
|  8|       16|
|  9|       18|
+---+---------+



In [9]:
df = spark.range(1, 100)
optimized = df.filter("id > 50").select("id")
optimized.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedalias('id, None)]
+- Filter (id#25L > cast(50 as bigint))
   +- Range (1, 100, step=1, splits=Some(3))

== Analyzed Logical Plan ==
id: bigint
Project [id#25L]
+- Filter (id#25L > cast(50 as bigint))
   +- Range (1, 100, step=1, splits=Some(3))

== Optimized Logical Plan ==
Filter (id#25L > 50)
+- Range (1, 100, step=1, splits=Some(3))

== Physical Plan ==
*(1) Filter (id#25L > 50)
+- *(1) Range (1, 100, step=1, splits=3)



In [10]:
df = spark.range(1, 100)
optimized = df.filter("id > 50").select("id")
optimized.explain(mode='extended')

== Parsed Logical Plan ==
'Project [unresolvedalias('id, None)]
+- Filter (id#30L > cast(50 as bigint))
   +- Range (1, 100, step=1, splits=Some(3))

== Analyzed Logical Plan ==
id: bigint
Project [id#30L]
+- Filter (id#30L > cast(50 as bigint))
   +- Range (1, 100, step=1, splits=Some(3))

== Optimized Logical Plan ==
Filter (id#30L > 50)
+- Range (1, 100, step=1, splits=Some(3))

== Physical Plan ==
*(1) Filter (id#30L > 50)
+- *(1) Range (1, 100, step=1, splits=3)



In [14]:
df = spark.range(1, 1000000)
df_filtered = df.filter(df.id % 2 == 0)
df_filtered.explain(True)

== Parsed Logical Plan ==
'Filter ((id#59L % 2) = 0)
+- Range (1, 1000000, step=1, splits=Some(3))

== Analyzed Logical Plan ==
id: bigint
Filter ((id#59L % cast(2 as bigint)) = cast(0 as bigint))
+- Range (1, 1000000, step=1, splits=Some(3))

== Optimized Logical Plan ==
Filter ((id#59L % 2) = 0)
+- Range (1, 1000000, step=1, splits=Some(3))

== Physical Plan ==
*(1) Filter ((id#59L % 2) = 0)
+- *(1) Range (1, 1000000, step=1, splits=3)



In [15]:
df = spark.range(1, 10)
result = df.filter("id % 2 == 0").groupBy().count()
result.explain()

== Physical Plan ==
*(2) HashAggregate(keys=[], functions=[count(1)])
+- Exchange SinglePartition, ENSURE_REQUIREMENTS, [id=#106]
   +- *(1) HashAggregate(keys=[], functions=[partial_count(1)])
      +- *(1) Project
         +- *(1) Filter ((id#63L % 2) = 0)
            +- *(1) Range (1, 10, step=1, splits=3)




In [16]:
df = spark.range(1, 10)

# Narrow
narrow_df = df.filter(df.id < 5)

#Wide
wide_df = df.groupBy((df.id % 2).alias("parity")).count()

narrow_df.show()
wide_df.show()

+---+
| id|
+---+
|  1|
|  2|
|  3|
|  4|
+---+

+------+-----+
|parity|count|
+------+-----+
|     0|    4|
|     1|    5|
+------+-----+



In [17]:
df = spark.range(1, 20).repartition(4)
print("Partitions:", df.rdd.getNumPartitions())

Partitions: 4


In [18]:
df = spark.range(1, 10)
shuffled = df.groupBy((df.id % 3).alias("grp")).count()
shuffled.explain()

== Physical Plan ==
*(2) HashAggregate(keys=[(id#104L % 3)#112L], functions=[count(1)])
+- Exchange hashpartitioning((id#104L % 3)#112L, 200), ENSURE_REQUIREMENTS, [id=#170]
   +- *(1) HashAggregate(keys=[(id#104L % 3) AS (id#104L % 3)#112L], functions=[partial_count(1)])
      +- *(1) Range (1, 10, step=1, splits=3)




In [19]:
df = spark.range(1, 100)

# Cache
df_cache = df.cache()

# Persist
from pyspark import StorageLevel
df_persisted = df.persist(StorageLevel.MEMORY_AND_DISK)

In [20]:
from pyspark.sql.functions import broadcast

large_df = spark.range(1, 1000000).withColumnRenamed("id", "key")
small_df = spark.createDataFrame([(1,),(2,),(3,)], ["key"])

joined = large_df.join(broadcast(small_df), "key")
joined.explain()

== Physical Plan ==
*(2) Project [key#124L]
+- *(2) BroadcastHashJoin [key#124L], [key#126L], Inner, BuildRight, false
   :- *(2) Project [id#122L AS key#124L]
   :  +- *(2) Range (1, 1000000, step=1, splits=3)
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [id=#205]
      +- *(1) Filter isnotnull(key#126L)
         +- *(1) Scan ExistingRDD[key#126L]




In [21]:
nobel_ds = "/Users/abhinavmohanty/Documents/Python/DataCamp/Data Scientist/Projects/A Visual History of Nobel Prize Winners/datasets/nobel.csv"
df_inferred = spark.read.csv(nobel_ds, header=True, inferSchema=True)
df_inferred.printSchema()

df_inferred.groupBy("category").count().show()

root
 |-- year: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- prize: string (nullable = true)
 |-- motivation: string (nullable = true)
 |-- prize_share: string (nullable = true)
 |-- laureate_id: string (nullable = true)
 |-- laureate_type: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- birth_city: string (nullable = true)
 |-- birth_country: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- organization_name: string (nullable = true)
 |-- organization_city: string (nullable = true)
 |-- organization_country: string (nullable = true)
 |-- death_date: string (nullable = true)
 |-- death_city: string (nullable = true)
 |-- death_country: string (nullable = true)

+----------+-----+
|  category|count|
+----------+-----+
| Chemistry|  175|
|  Medicine|  211|
|   Physics|  204|
|Literature|  113|
| Economics|   78|
|     Peace|  130|
+----------+-----+



In [22]:
# DataFrome Schema Aware
df = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])
df.show()

rdd = df.rdd.map(lambda row: (row[0], row[1].upper()))
print(rdd.collect())

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+

[(1, 'ALICE'), (2, 'BOB')]


In [23]:
# create DataFrame
data = [("Alice", 23), ("Bob", 29), ("Charlie", 37)]
df = spark.createDataFrame(data, ["name", "age"])

# Register DataFrame as temporary SQL view
df.createOrReplaceTempView("people")

# Run SQL query
sql_result = spark.sql("Select name, age from people where age > 25")

sql_result.show()

+-------+---+
|   name|age|
+-------+---+
|    Bob| 29|
|Charlie| 37|
+-------+---+



In [24]:
data = [("Alice", 23), ("Bob", 29)]
df = spark.createDataFrame(data, ["name", "age"])

result = df.filter("age > 20")
result.show()

+-----+---+
| name|age|
+-----+---+
|Alice| 23|
|  Bob| 29|
+-----+---+



In [25]:
from pyspark.sql.functions import udf

In [26]:
# Sample sales DataFrame
data = [("A", 100), ("B", 200), ("C", 150), ("A", 120), ("C", 250), ("B", 160), ("A", 170)]
columns = ["product_id", "amount"]

sales_df = spark.createDataFrame(data, columns)

# Lookup dictionary for Categories
product_lookup = {"A": "Electronics", "B": "Clothing", "C": "Books"}

# Broadcast the lookup
broadcast_lookup = spark.sparkContext.broadcast(product_lookup)

@udf
def get_category(product_id):
    return broadcast_lookup.value.get(product_id, "Unknown")

# Add category column to the DataFrame
sales_df = sales_df.withColumn("category", get_category("product_id"))
sales_df.show()

+----------+------+-----------+
|product_id|amount|   category|
+----------+------+-----------+
|         A|   100|Electronics|
|         B|   200|   Clothing|
|         C|   150|      Books|
|         A|   120|Electronics|
|         C|   250|      Books|
|         B|   160|   Clothing|
|         A|   170|Electronics|
+----------+------+-----------+



In [28]:
data = [("US", 100), ("IN", 200), ("US", 50), ("IN", 75)]
rdd = spark.sparkContext.parallelize(data)

# Ex reduceByKey to sum sales
result = rdd.reduceByKey(lambda a, b: a + b).collect()
print(result)

[('US', 150), ('IN', 275)]


In [29]:
# Sample k-v RDD
data = [("apple", 1), ("banana", 3), ("apple", 5), ("banana", 7), ("orange", 11)]
rdd = spark.sparkContext.parallelize(data)

# groupByKey
grouped = rdd.groupByKey()

# Collect Result
for k, v in grouped.collect():
    print(k, list(v))

orange [11]
banana [3, 7]
apple [1, 5]


In [30]:
# Sum of values for each fruit
reduced = rdd.reduceByKey( lambda x, y: x + y)

print(reduced.collect())

[('orange', 11), ('banana', 10), ('apple', 6)]


In [31]:
df = spark.range(0, 20).repartition(6) # create 6 partitions
print("Partitions Before:", df.rdd.getNumPartitions())

df2 = df.coalesce(2) # reduce to 2 partitions without shuffle
print("Partitions After:", df2.rdd.getNumPartitions())

df3 = df2.repartition(10) # increase to 10 partitions with shuffle
print("Partitions after repartitions:", df3.rdd.getNumPartitions())

Partitions Before: 6
Partitions After: 2
Partitions after repartitions: 10


In [3]:
data = [(1, 'Alice', 'DE'), (2, 'Bob', 'SE'), (3, 'Charlie', 'DA'), (4, 'David', 'DA')]
schema = 'id int, name string, designation string'
df = spark.createDataFrame(data, schema)
display(df)

DataFrame[id: int, name: string, designation: string]

In [4]:
df.show()

+---+-------+-----------+
| id|   name|designation|
+---+-------+-----------+
|  1|  Alice|         DE|
|  2|    Bob|         SE|
|  3|Charlie|         DA|
|  4|  David|         DA|
+---+-------+-----------+



In [5]:
df.write.format('delta').mode('overwrite').saveAsTable('employee_detail')

Py4JJavaError: An error occurred while calling o50.saveAsTable.
: java.lang.ClassNotFoundException: Failed to find data source: delta. Please find packages at http://spark.apache.org/third-party-projects.html
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:689)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSourceV2(DataSource.scala:743)
	at org.apache.spark.sql.DataFrameWriter.lookupV2Provider(DataFrameWriter.scala:993)
	at org.apache.spark.sql.DataFrameWriter.saveAsTable(DataFrameWriter.scala:615)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:566)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4

In [6]:
df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true') \
    .save('/Users/abhinavmohanty/Documents/Python')

Py4JJavaError: An error occurred while calling o56.save.
: java.lang.ClassNotFoundException: Failed to find data source: delta. Please find packages at http://spark.apache.org/third-party-projects.html
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:689)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSourceV2(DataSource.scala:743)
	at org.apache.spark.sql.DataFrameWriter.lookupV2Provider(DataFrameWriter.scala:993)
	at org.apache.spark.sql.DataFrameWriter.saveInternal(DataFrameWriter.scala:311)
	at org.apache.spark.sql.DataFrameWriter.save(DataFrameWriter.scala:293)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:566)
	at p

In [8]:
df.write.format('parquet').mode('overwrite').option('overwriteSchema', 'true') \
    .save('/Users/abhinavmohanty/Documents/Python/test_parquet')